In [ ]:
# Cell 1 — imports
import re
import redivis
import pandas as pd

MIN_COMPANY_NAME_LEN = 4

In [ ]:
# Cell 2 — load our URL list from the uploaded dataset
urls_table = redivis.user("ml2068").dataset("urls_for_redivis:5n0r").table("urls_for_redivis:8nyr")
urls_df = urls_table.to_pandas_dataframe()
print(f"Our URLs: {len(urls_df):,} rows")
urls_df.head(3)

In [ ]:
# Cell 3 — join URLs → individual_user (server-side, only matched rows returned)
matched_users_df = redivis.query("""
    SELECT
        u.user_id,
        u.firstname,
        u.lastname,
        u.fullname,
        u.profile_linkedin_url,
        u.profile_title,
        u.numconnections,
        u.user_country,
        u.prestige,
        urls.clean_linkedin_url
    FROM `urls_for_redivis:8nyr` AS urls
    INNER JOIN `individual_user:xcsm` AS u
        ON urls.clean_linkedin_url = u.profile_linkedin_url
""").to_pandas_dataframe()

print(f"Matched users: {len(matched_users_df):,} rows")
matched_users_df.head(3)

In [ ]:
# Cell 4 — fetch positions via JOIN (avoids IN clause size limit)
positions_df = redivis.query("""
    SELECT
        p.user_id,
        p.company_cleaned,
        p.seniority,
        p.startdate,
        p.enddate
    FROM `individual_position:8xgp` AS p
    INNER JOIN `matched_users:kh5e` AS m
        ON p.user_id = m.user_id
""").to_pandas_dataframe()

print(f"Positions fetched: {len(positions_df):,} rows")
positions_df.head(3)

In [ ]:
# Cell 5 — build positions index {user_id: [company_cleaned, ...]}
positions_index = {}
for row in positions_df[["user_id", "company_cleaned"]].itertuples(index=False):
    uid = row.user_id
    if uid not in positions_index:
        positions_index[uid] = []
    positions_index[uid].append(row.company_cleaned)

print(f"Position index built for {len(positions_index):,} users")

In [ ]:
# Cell 6 — helper functions
import unicodedata
from difflib import SequenceMatcher

SUFFIXES = {
    "jr", "sr", "ii", "iii", "iv",
    "phd", "ph.d", "ph.d.", "md", "m.d", "m.d.",
    "jd", "j.d", "j.d.", "mba", "m.b.a", "cpa", "cfa", "esq",
    "ba", "ma", "ms", "bs", "mpa", "mph",
    "icd.d", "icd", "ret", "retired",
    "usaf", "usa", "usmc", "usn", "uscg", "cm",
}

LEGAL_SUFFIXES = re.compile(
    r"\b(inc|llc|corp|corporation|ltd|limited|gmbh|co|group|plc|sa|ag|bv|nv|lp|llp|holdings|holding|international|intl)\b",
    re.IGNORECASE
)


def to_ascii(s):
    """Normalize accents and special chars → plain ASCII lowercase."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9\s]", "", s.lower()).strip()


def clean_person_name(name):
    """Strip credentials/suffixes and return cleaned name tokens."""
    if pd.isna(name):
        return []
    name = str(name).split(",")[0].strip()
    tokens = name.lower().split()
    while tokens and tokens[-1].rstrip(".") in SUFFIXES:
        tokens.pop()
    return tokens


def name_matches(revelio_fullname, our_name, our_name_clean=None):
    """Four-stage cascade: last name → hyphen → first+initial → difflib."""
    if pd.isna(revelio_fullname):
        return False
    rev = to_ascii(revelio_fullname)

    for name in [our_name_clean, our_name]:
        tokens = clean_person_name(name)
        if not tokens:
            continue
        last = to_ascii(tokens[-1])
        first = to_ascii(tokens[0]) if len(tokens) > 1 else ""

        # Stage 1: last name substring
        if last and last in rev:
            return True

        # Stage 2: hyphenated last name — check each part
        if "-" in last:
            if any(part in rev for part in last.split("-") if part):
                return True

        # Stage 3: first name present + last initial present
        if first and last and first in rev and last[0] in rev:
            return True

        # Stage 4: difflib similarity on full name
        our_full = to_ascii(" ".join(tokens))
        if our_full and SequenceMatcher(None, our_full, rev).ratio() >= 0.85:
            return True

    return False


def strip_legal(name):
    """Strip legal suffixes and normalize to ASCII."""
    return to_ascii(LEGAL_SUFFIXES.sub("", str(name))).strip()


def company_in_positions(user_id, company_name, company_name_orig=None):
    """Bidirectional substring + token overlap after legal suffix stripping."""
    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < MIN_COMPANY_NAME_LEN:
            continue
        our = strip_legal(name)
        our_tokens = set(our.split()) - {""}
        if not our_tokens:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            # Bidirectional substring
            if our in pos_clean or pos_clean in our:
                return True
            # Token overlap: ≥50% of our tokens appear in position string
            pos_tokens = set(pos_clean.split())
            overlap = our_tokens & pos_tokens
            if len(overlap) / len(our_tokens) >= 0.5:
                return True
    return False


def clean_url(url):
    """Normalise to linkedin.com/in/<slug>."""
    if pd.isna(url):
        return None
    url = str(url).strip().rstrip("/")
    url = re.sub(r"^https?://(www\.)?", "", url)
    return url if url.startswith("linkedin.com/in/") else None

# ──────────────────────────────────────────────────────────
# Fuzzy company matcher — catches parent/subsidiary, abbreviations,
# and short names that the strict matcher misses. Computed in
# PARALLEL with the strict version so we can compare directly.
# ──────────────────────────────────────────────────────────

FUZZY_MIN_COMPANY_LEN = 3   # allow IBM, GE, HP, AT&T (vs strict's 4)
FUZZY_RATIO_THRESHOLD = 0.80


def company_in_positions_fuzzy(user_id, company_name, company_name_orig=None):
    """Strict match first; if that fails, try difflib similarity ≥ 0.80."""
    if company_in_positions(user_id, company_name, company_name_orig):
        return True

    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < FUZZY_MIN_COMPANY_LEN:
            continue
        our = strip_legal(name)
        if not our or len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            if not pos_clean or len(pos_clean) < FUZZY_MIN_COMPANY_LEN:
                continue
            if SequenceMatcher(None, our, pos_clean).ratio() >= FUZZY_RATIO_THRESHOLD:
                return True
    return False


In [ ]:
# Cell 7 — build lookup: clean_url → revelio user row
revelio_by_url = matched_users_df.drop_duplicates("clean_linkedin_url").set_index("clean_linkedin_url")

In [ ]:
# Cell 8 — load all_linkedin_urls (uploaded as dataset 'all_linkedin_urls')
all_urls_table = redivis.user("ml2068").dataset("all_linkedin_urls").table("all_linkedin_urls")
all_people_df = all_urls_table.to_pandas_dataframe(
    variables=["person_name", "person_name_clean", "company_name", "company_name_clean",
               "source", "linkedin_url", "verified", "gvkey", "ticker", "is_entity"]
)
print(f"all_linkedin_urls: {len(all_people_df):,} rows")
all_people_df.head(3)

In [ ]:
# Cell 9 — normalise URLs for join
all_people_df["clean_url"] = all_people_df["linkedin_url"].apply(clean_url)

In [ ]:
# Cell 10 — compute confirmation columns
# Computes BOTH strict and fuzzy company-match in parallel so we can
# quantify how many additional valid matches the looser rule recovers.

revelio_url_match = []
revelio_name_confirmed = []
revelio_company_confirmed = []         # strict
revelio_company_confirmed_fuzzy = []   # NEW: looser company match
revelio_user_id_col = []

all_people_df = all_people_df.rename(columns={"_clean_url": "clean_url"}) if "_clean_url" in all_people_df.columns else all_people_df

for row in all_people_df.itertuples(index=False):
    clean = row.clean_url
    rev = revelio_by_url.loc[clean] if (clean and clean in revelio_by_url.index) else None

    if rev is None:
        revelio_url_match.append(False)
        revelio_name_confirmed.append(False)
        revelio_company_confirmed.append(False)
        revelio_company_confirmed_fuzzy.append(False)
        revelio_user_id_col.append(None)
    else:
        revelio_url_match.append(True)
        uid = rev["user_id"]
        revelio_user_id_col.append(uid)
        revelio_name_confirmed.append(
            name_matches(rev["fullname"], row.person_name, getattr(row, "person_name_clean", None))
        )
        co_orig = getattr(row, "company_name", None)
        revelio_company_confirmed.append(
            company_in_positions(uid, row.company_name_clean, co_orig)
        )
        revelio_company_confirmed_fuzzy.append(
            company_in_positions_fuzzy(uid, row.company_name_clean, co_orig)
        )

all_people_df["revelio_url_match"] = revelio_url_match
all_people_df["revelio_name_confirmed"] = revelio_name_confirmed
all_people_df["revelio_company_confirmed"] = revelio_company_confirmed
all_people_df["revelio_company_confirmed_fuzzy"] = revelio_company_confirmed_fuzzy
all_people_df["revelio_user_id"] = revelio_user_id_col

print("Done.")


In [ ]:
# Cell 11 — summary stats (strict vs fuzzy company match)
total = len(all_people_df)
found = all_people_df["clean_url"].notna().sum()
matched = sum(revelio_url_match)
name_conf = sum(revelio_name_confirmed)
co_conf_strict = sum(revelio_company_confirmed)
co_conf_fuzzy = sum(revelio_company_confirmed_fuzzy)
both_strict = sum(a and b for a, b in zip(revelio_name_confirmed, revelio_company_confirmed))
both_fuzzy = sum(a and b for a, b in zip(revelio_name_confirmed, revelio_company_confirmed_fuzzy))

verified = all_people_df["verified"].fillna(False).astype(bool).tolist()
strong_strict = sum(
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified, revelio_company_confirmed)
)
strong_fuzzy = sum(
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified, revelio_company_confirmed_fuzzy)
)

print(f"Total rows:                        {total:>8,}")
print(f"Has URL:                           {found:>8,}")
print(f"Revelio URL match:                 {matched:>8,}  ({matched/found*100:.1f}% of found)")
print(f"  Name confirmed:                  {name_conf:>8,}  ({name_conf/matched*100:.1f}%)")
print()
print(f"Company confirmed — STRICT:        {co_conf_strict:>8,}  ({co_conf_strict/matched*100:.1f}%)")
print(f"Company confirmed — FUZZY:         {co_conf_fuzzy:>8,}  ({co_conf_fuzzy/matched*100:.1f}%)")
print(f"  Δ recovered by fuzzy:            {co_conf_fuzzy-co_conf_strict:>8,}")
print()
print(f"Strong match — STRICT:             {strong_strict:>8,}  ({strong_strict/matched*100:.1f}%)")
print(f"Strong match — FUZZY:              {strong_fuzzy:>8,}  ({strong_fuzzy/matched*100:.1f}%)")
print(f"  Δ recovered by fuzzy:            {strong_fuzzy-strong_strict:>8,}")


In [ ]:
# Cell 12 — export summary with BOTH strict and fuzzy company-match results
verified_s = all_people_df["verified"].fillna(False).astype(bool)

# Strict (current production rule, kept as the primary `strong_match`)
all_people_df["strong_match"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, revelio_company_confirmed)
]

# Fuzzy (looser company match — restored to capture parent/subsidiary,
# abbreviations, and short company names)
all_people_df["strong_match_fuzzy"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, revelio_company_confirmed_fuzzy)
]

output = all_people_df[[
    "linkedin_url",
    "person_name",
    "company_name_clean",
    "source",
    "revelio_url_match",
    "revelio_name_confirmed",
    "revelio_company_confirmed",          # strict (original)
    "revelio_company_confirmed_fuzzy",    # NEW
    "revelio_user_id",
    "strong_match",                       # strict (original)
    "strong_match_fuzzy",                 # NEW
]].copy()

output.to_csv("revelio_validation_summary.csv", index=False)
print(f"Exported revelio_validation_summary.csv ({len(output):,} rows)")
print(f"  Strong matches (strict): {output['strong_match'].sum():,} "
      f"({output['strong_match'].mean()*100:.1f}%)")
print(f"  Strong matches (fuzzy):  {output['strong_match_fuzzy'].sum():,} "
      f"({output['strong_match_fuzzy'].mean()*100:.1f}%)")
print(f"  Δ recovered by fuzzy:    {(output['strong_match_fuzzy'].sum() - output['strong_match'].sum()):,}")

redivis.current_notebook().create_output_table(output)
print("Output table created in Redivis workflow.")


In [ ]:
# Cell 12.5 — DIAGNOSTIC (print only, no export)
# For every row where strict company-match failed but fuzzy succeeded,
# replay the fuzzy match to find which Revelio position string and ratio
# triggered it. Print top-30 (best matches — should be obvious variants)
# and bottom-30 (marginal — most likely false positives).
# Use this to decide whether the +708 are signal or noise.

rescued = []  # list of (ratio, our_company, matched_position, source)

for i, (strict, fuzzy) in enumerate(zip(revelio_company_confirmed,
                                        revelio_company_confirmed_fuzzy)):
    if not (fuzzy and not strict):
        continue
    uid = revelio_user_id_col[i]
    if uid is None:
        continue
    row = all_people_df.iloc[i]
    co_clean = row.get("company_name_clean")
    co_orig  = row.get("company_name") if "company_name" in all_people_df.columns else None
    src      = row.get("source", "?")

    positions = positions_index.get(int(uid), [])
    best = (0.0, None)
    for name_input in [co_clean, co_orig]:
        if pd.isna(name_input) or len(str(name_input)) < FUZZY_MIN_COMPANY_LEN:
            continue
        our = strip_legal(name_input)
        if not our or len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            if not pos_clean or len(pos_clean) < FUZZY_MIN_COMPANY_LEN:
                continue
            r = SequenceMatcher(None, our, pos_clean).ratio()
            if r >= FUZZY_RATIO_THRESHOLD and r > best[0]:
                best = (r, pos)
    if best[1] is not None:
        rescued.append((best[0], co_clean, best[1], src))

print(f"Total rescued by fuzzy: {len(rescued):,}")
print()
print("=" * 90)
print("TOP 30  (highest ratio — these should look like obvious variants)")
print("=" * 90)
print(f"  {'ratio':>5}  {'src':<10}  {'our company':<35}  →  Revelio position")
for r, our, pos, src in sorted(rescued, reverse=True)[:30]:
    print(f"  {r:>5.2f}  {str(src)[:10]:<10}  {str(our)[:35]:<35}  →  {pos}")

print()
print("=" * 90)
print("BOTTOM 30  (lowest ratio — most marginal, watch for false positives)")
print("=" * 90)
print(f"  {'ratio':>5}  {'src':<10}  {'our company':<35}  →  Revelio position")
for r, our, pos, src in sorted(rescued)[:30]:
    print(f"  {r:>5.2f}  {str(src)[:10]:<10}  {str(our)[:35]:<35}  →  {pos}")

print()
print("=" * 90)
print("RANDOM 20  (typical case)")
print("=" * 90)
import random
random.seed(42)
sample = random.sample(rescued, min(20, len(rescued)))
print(f"  {'ratio':>5}  {'src':<10}  {'our company':<35}  →  Revelio position")
for r, our, pos, src in sample:
    print(f"  {r:>5.2f}  {str(src)[:10]:<10}  {str(our)[:35]:<35}  →  {pos}")


In [ ]:
# Cell 13 — S&P 500 coverage analysis
sp500_table = redivis.user("ml2068").dataset("sp500").table("sp500_companies")
sp500_df = sp500_table.to_pandas_dataframe(variables=["gvkey", "ticker", "company_name"])

def norm_gvkey(x):
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return None

sp500_gvkeys = set(norm_gvkey(g) for g in sp500_df["gvkey"].dropna())
sp500_gvkeys.discard(None)
print(f"S&P 500 companies: {len(sp500_gvkeys):,} unique gvkeys")

# Normalise gvkeys + tag S&P 500
all_people_df["gvkey_norm"] = all_people_df["gvkey"].apply(norm_gvkey)
all_people_df["is_sp500"] = all_people_df["gvkey_norm"].isin(sp500_gvkeys)

# Compute signal columns
verified_s = all_people_df["verified"].fillna(False).astype(bool)
all_people_df["strong_match"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, revelio_company_confirmed)
]
all_people_df["revelio_url_match_col"] = revelio_url_match
all_people_df["verified_bool"] = verified_s

# People only (exclude entity blockholders)
people_df = all_people_df[all_people_df["is_entity"] == False].copy()
print(f"People rows (is_entity=False): {len(people_df):,}")

sp500_p = people_df[people_df["is_sp500"]]
non_sp500_p = people_df[~people_df["is_sp500"]]

# ── Helper ──────────────────────────────────────────────────────────────
def coverage_stats(df, label):
    total = len(df)
    has_url = df["linkedin_url"].notna().sum()
    verified = df["verified_bool"].sum()
    rev_match = df["revelio_url_match_col"].sum()
    strong = df["strong_match"].sum()
    print(f"{label} (n={total:,}):")
    print(f"  Has URL:            {has_url:>8,}  ({has_url/total*100:.1f}% of people)")
    print(f"  Verified (name):    {verified:>8,}  ({verified/total*100:.1f}% of people)")
    print(f"  Revelio matched:    {rev_match:>8,}  ({rev_match/has_url*100:.1f}% of URLs found)")
    print(f"  Strong match:       {strong:>8,}  ({strong/rev_match*100:.1f}% of Revelio matched)")
    print()

coverage_stats(sp500_p,     "S&P 500 companies")
coverage_stats(non_sp500_p, "Non-S&P 500 companies")
coverage_stats(people_df,   "All people")

# ── Source breakdown ─────────────────────────────────────────────────────
print("Strong match rate by source (people only):")
for src, grp in people_df.groupby("source"):
    rev = grp["revelio_url_match_col"].sum()
    strong = grp["strong_match"].sum()
    rate = strong / rev * 100 if rev else 0
    print(f"  {src:<35} n={len(grp):>6,}  strong={strong:>5,}  ({rate:.1f}% of Revelio matched)")
print()

# ── Company-level coverage distribution ──────────────────────────────────
company_level = people_df.groupby(["gvkey_norm", "company_name", "is_sp500"]).agg(
    total_people   = ("strong_match", "count"),
    has_url        = ("linkedin_url", lambda x: x.notna().sum()),
    rev_matched    = ("revelio_url_match_col", "sum"),
    strong         = ("strong_match", "sum"),
    verified       = ("verified_bool", "sum"),
).reset_index()

print("Company-level strong match distribution (people only):")
for label, grp in [("S&P 500", company_level[company_level["is_sp500"]]),
                   ("Non-S&P 500", company_level[~company_level["is_sp500"]]),
                   ("All", company_level)]:
    total_cos = len(grp)
    zero      = (grp["strong"] == 0).sum()
    at_least1 = (grp["strong"] >= 1).sum()
    at_least2 = (grp["strong"] >= 2).sum()
    at_least5 = (grp["strong"] >= 5).sum()
    median    = grp["strong"].median()
    mean      = grp["strong"].mean()
    print(f"  {label} ({total_cos:,} companies):")
    print(f"    0 strong matches:     {zero:>5,}  ({zero/total_cos*100:.1f}%)")
    print(f"    ≥1 strong match:      {at_least1:>5,}  ({at_least1/total_cos*100:.1f}%)")
    print(f"    ≥2 strong matches:    {at_least2:>5,}  ({at_least2/total_cos*100:.1f}%)")
    print(f"    ≥5 strong matches:    {at_least5:>5,}  ({at_least5/total_cos*100:.1f}%)")
    print(f"    Median per company:   {median:.1f}")
    print(f"    Mean per company:     {mean:.1f}")
    print()